In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.ensemble import StackingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OrdinalEncoder
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')

# 1. Chargement des Données

In [2]:
print("Chargement des données...")
try:
    # Chargement des données (on suppose que les fichiers sont dans le répertoire courant)
    X_train = pd.read_csv('X_train.csv', index_col=0, low_memory=False)
    y_train = pd.read_csv('y_train.csv', index_col=0, low_memory=False)
    X_test = pd.read_csv('X_test.csv', index_col=0, low_memory=False)
    
    print(f"X_train : {X_train.shape}")
    print(f"y_train : {y_train.shape}")
    print(f"X_test  : {X_test.shape}")
    
    # Alignement des index train/target
    common_indices = X_train.index.intersection(y_train.index)
    X_train = X_train.loc[common_indices]
    y_train = y_train.loc[common_indices]
    
except FileNotFoundError:
    print("Erreur : Fichiers introuvables. Vérifiez le chemin.")

Chargement des données...
X_train : (1172086, 306)
y_train : (1172086, 1)
X_test  : (586044, 306)
X_train : (1172086, 306)
y_train : (1172086, 1)
X_test  : (586044, 306)


# 2. Preprocessing & Feature Selection Optimisée

In [3]:
print("--- Nettoyage et Sélection des Features ---")

# 1. Suppression des colonnes avec > 50% de NaN
missing_threshold = 0.5
missing_ratio = X_train.isnull().mean()
cols_to_drop_nan = missing_ratio[missing_ratio > missing_threshold].index.tolist()
print(f"Suppression de {len(cols_to_drop_nan)} colonnes ayant > {missing_threshold*100}% de valeurs manquantes.")

X_train_clean = X_train.drop(columns=cols_to_drop_nan)

# 2. Alignement des colonnes Train / Test
# On ne garde que les colonnes présentes dans les deux (et qui n'ont pas été supprimées)
common_cols = X_train_clean.columns.intersection(X_test.columns)

# Filtrage des colonnes 'math_' qui pourraient fuiter (data leakage)
final_features = [c for c in common_cols if not c.lower().startswith('math_')]

print(f"Nombre final de features conservées : {len(final_features)}")

X_train_final = X_train_clean[final_features].copy()
X_test_final = X_test[final_features].copy()

# 3. Gestion des types pour CatBoost
# CatBoost gère nativement les catégories, mais il faut lui dire lesquelles c'est.
# Il faut aussi s'assurer qu'il n'y a pas de NaN dans les colonnes catégorielles (CatBoost préfère une valeur 'Missing')

cat_features = X_train_final.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Features catégorielles identifiées : {len(cat_features)}")

# Remplissage des NaN dans les variables catégorielles par "MISSING"
# Pour les numériques, CatBoost gère les NaN nativement, on laisse tel quel.
for col in cat_features:
    X_train_final[col] = X_train_final[col].fillna("MISSING").astype(str)
    X_test_final[col] = X_test_final[col].fillna("MISSING").astype(str)

print("Preprocessing terminé.")

--- Nettoyage et Sélection des Features ---
Suppression de 206 colonnes ayant > 50.0% de valeurs manquantes.
Suppression de 206 colonnes ayant > 50.0% de valeurs manquantes.
Nombre final de features conservées : 82
Nombre final de features conservées : 82
Features catégorielles identifiées : 3
Preprocessing terminé.
Features catégorielles identifiées : 3
Preprocessing terminé.


# 3. Entraînement CatBoost Optimisé

In [4]:
# Split Train/Validation (80/20) pour l'early stopping
X_tr, X_val, y_tr, y_val = train_test_split(X_train_final, y_train, test_size=0.2, random_state=42)

print(f"Train shape: {X_tr.shape}, Val shape: {X_val.shape}")

# Configuration optimisée de CatBoost
model = CatBoostRegressor(
    iterations=2000,          # Nombre élevé d'arbres, l'early stopping arrêtera avant si nécessaire
    learning_rate=0.05,       # Learning rate modéré pour la précision
    depth=8,                  # Profondeur 8 est un bon compromis performance/vitesse (vs 6 ou 10)
    loss_function='RMSE',
    eval_metric='R2',
    random_seed=42,
    verbose=100,              # Affiche la progression tous les 100 arbres
    early_stopping_rounds=100, # Arrête si le score ne s'améliore plus pendant 100 itérations
    cat_features=cat_features # Gestion native des catégories
)

print("Lancement de l'entraînement avec Early Stopping...")
model.fit(
    X_tr, y_tr,
    eval_set=(X_val, y_val),
    use_best_model=True       # Garde le modèle qui a eu le meilleur score sur la validation
)

print("Entraînement terminé.")

Train shape: (937668, 82), Val shape: (234418, 82)
Lancement de l'entraînement avec Early Stopping...
0:	learn: 0.0382514	test: 0.0387613	best: 0.0387613 (0)	total: 728ms	remaining: 24m 14s
0:	learn: 0.0382514	test: 0.0387613	best: 0.0387613 (0)	total: 728ms	remaining: 24m 14s
100:	learn: 0.4814566	test: 0.4835953	best: 0.4835953 (100)	total: 1m 23s	remaining: 26m 2s
100:	learn: 0.4814566	test: 0.4835953	best: 0.4835953 (100)	total: 1m 23s	remaining: 26m 2s
200:	learn: 0.4918794	test: 0.4930862	best: 0.4930862 (200)	total: 2m 46s	remaining: 24m 49s
200:	learn: 0.4918794	test: 0.4930862	best: 0.4930862 (200)	total: 2m 46s	remaining: 24m 49s
300:	learn: 0.4967766	test: 0.4971817	best: 0.4971817 (300)	total: 3m 54s	remaining: 22m 6s
300:	learn: 0.4967766	test: 0.4971817	best: 0.4971817 (300)	total: 3m 54s	remaining: 22m 6s
400:	learn: 0.5005759	test: 0.5000429	best: 0.5000429 (400)	total: 5m 4s	remaining: 20m 15s
400:	learn: 0.5005759	test: 0.5000429	best: 0.5000429 (400)	total: 5m 4s	rem

# 4. Évaluation

In [5]:
# Prédictions sur la validation
y_pred_val = model.predict(X_val)
y_pred_val = np.clip(y_pred_val, 0, 1000) # Clipping métier PISA

r2 = r2_score(y_val, y_pred_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))

print(f"=== RÉSULTATS VALIDATION ===")
print(f"Meilleure itération : {model.get_best_iteration()}")
print(f"R²   : {r2:.5f}")
print(f"RMSE : {rmse:.5f}")

=== RÉSULTATS VALIDATION ===
Meilleure itération : 1998
R²   : 0.50979
RMSE : 85.44731


# 5. Ré-entraînement Final (Optionnel mais recommandé)
Pour maximiser la performance, on peut ré-entraîner sur TOUT le dataset (Train + Val) en utilisant le nombre optimal d'itérations trouvé précédemment.

In [6]:
print("Ré-entraînement sur l'ensemble du dataset avec le nombre optimal d'itérations...")

best_iter = model.get_best_iteration()

final_model = CatBoostRegressor(
    iterations=best_iter,     # On utilise le nombre optimal trouvé
    learning_rate=0.05,
    depth=8,
    loss_function='RMSE',
    random_seed=42,
    verbose=100,
    cat_features=cat_features,
    allow_writing_files=False
)

final_model.fit(X_train_final, y_train)
print("Modèle final prêt.")

Ré-entraînement sur l'ensemble du dataset avec le nombre optimal d'itérations...
0:	learn: 119.8037436	total: 604ms	remaining: 20m 6s
0:	learn: 119.8037436	total: 604ms	remaining: 20m 6s
100:	learn: 87.9453337	total: 1m 27s	remaining: 27m 19s
100:	learn: 87.9453337	total: 1m 27s	remaining: 27m 19s
200:	learn: 87.0573992	total: 3m 20s	remaining: 29m 51s
200:	learn: 87.0573992	total: 3m 20s	remaining: 29m 51s
300:	learn: 86.6513015	total: 4m 58s	remaining: 28m 1s
300:	learn: 86.6513015	total: 4m 58s	remaining: 28m 1s
400:	learn: 86.3306370	total: 6m 36s	remaining: 26m 20s
400:	learn: 86.3306370	total: 6m 36s	remaining: 26m 20s
500:	learn: 86.0779330	total: 8m 18s	remaining: 24m 50s
500:	learn: 86.0779330	total: 8m 18s	remaining: 24m 50s
600:	learn: 85.8728784	total: 10m	remaining: 23m 15s
600:	learn: 85.8728784	total: 10m	remaining: 23m 15s
700:	learn: 85.6915225	total: 11m 33s	remaining: 21m 23s
700:	learn: 85.6915225	total: 11m 33s	remaining: 21m 23s
800:	learn: 85.5288848	total: 13m 1

# 6. Prédiction et Soumission

In [ ]:
print("Génération des prédictions sur X_test...")
y_test_pred = final_model.predict(X_test_final)

# Clipping [0, 1000]
y_test_pred = np.clip(y_test_pred, 0, 1000)

print("Stats prédictions :")
print(pd.Series(y_test_pred).describe())

# Création fichier de soumission
try:
    sample_sub = pd.read_csv('sample_submission_hfactory.csv')
    pred_df = pd.DataFrame({'ID': X_test.index, 'MathScore': y_test_pred})

    if 'MathScore' in sample_sub.columns:
        del sample_sub['MathScore']

    final_submission = sample_sub.merge(pred_df, on='ID', how='left')

    # Remplissage manquants éventuels par la moyenne globale
    if final_submission['MathScore'].isnull().any():
        mean_val = y_train.values.ravel().mean()
        print(f"Remplissage de {final_submission['MathScore'].isnull().sum()} valeurs manquantes.")
        final_submission['MathScore'] = final_submission['MathScore'].fillna(mean_val)

    output_file = 'submission_catboost_optimized.csv'
    final_submission.to_csv(output_file, index=False)
    print(f"\nFichier généré : {output_file}")
    print(final_submission.head())
    
except FileNotFoundError:
    print("Erreur : sample_submission_hfactory.csv introuvable. Génération d'un fichier simple.")
    pred_df = pd.DataFrame({'ID': X_test.index, 'MathScore': y_test_pred})
    pred_df.to_csv('submission_catboost_simple.csv', index=False)
    print("Fichier submission_catboost_simple.csv généré.")